# Softmax 分类器 — MNIST 手写数字识别

用一个 5 层全连接网络识别手写数字（0-9，共 10 类）。下面按四个部分讲解代码。

---

## 1. 数据准备

```python
batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
```

- `Compose` — 把多个预处理步骤串成一条流水线。
- `ToTensor()` — 把图片（PIL，像素 0-255）转成张量并缩放到 0-1，形状变为 `(通道, 高, 宽)` = `(1, 28, 28)`。
- `Normalize((0.1307,), (0.3081,))` — 标准化 `(x - 均值) / 标准差`。这两个数是整个 MNIST 的均值和标准差（前人算好的）。标准化让数据分布更稳定，训练更快更好。

`datasets.MNIST(..., download=True)` 自动下载数据集；`train=True/False` 分别取训练集 / 测试集。`DataLoader` 负责分批 + 打乱：训练集 `shuffle=True`，测试集 `shuffle=False`（只评估，无需打乱）。

---

## 2. 模型设计

5 层全连接网络，维度逐层缩小：`784 → 512 → 256 → 128 → 64 → 10`。

- **输入 784** = 28 × 28（图片摊平成一维向量）。
- **输出 10** = 10 个类别，每个输出是该类别的 logit。

```python
def forward(self, x):
    x = x.view(-1, 784)        # (batch,1,28,28) 摊平成 (batch,784)，-1 自动算 batch 大小
    x = F.relu(self.l1(x))
    ...
    return self.l5(x)          # 最后一层不加激活
```

- 前四层用 **ReLU** 引入非线性。
- **最后一层不加激活**（关键）：输出原始 logits，因为后面的 `CrossEntropyLoss` 内部会自己做 softmax。这里再加 softmax 就重复了。

---

## 3. 损失函数和优化器

```python
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
```

- `CrossEntropyLoss` —— 多分类交叉熵，**内部 = softmax + log + NLL 损失**，所以模型只输出 logits 即可。
- `SGD` 带 `momentum=0.5` —— 动量让更新方向带"惯性"，收敛更快更稳。

### CrossEntropyLoss 与 NLLLoss 的区别

**`CrossEntropyLoss` = `LogSoftmax` + `NLLLoss`**，两者本质是同一计算的拆分：

| | 输入要求 | 内部操作 |
|---|---|---|
| `NLLLoss` | **log概率**（已经做过 `log_softmax` 的值） | 直接取出对应类别的负值，求平均 |
| `CrossEntropyLoss` | **原始 logits**（未归一化） | 先做 `log_softmax`，再做 `NLLLoss` |

```python
# 两种写法完全等价
criterion1 = torch.nn.CrossEntropyLoss()
loss1 = criterion1(outputs, target)          # outputs是logits (64,10)

criterion2 = torch.nn.NLLLoss()
log_probs = F.log_softmax(outputs, dim=1)
loss2 = criterion2(log_probs, target)        # 需要先手动log_softmax

# loss1 == loss2
```

二者对 `target` 的格式要求相同：都是类别索引（如 `7`），不是 one-hot。本 notebook 用 `CrossEntropyLoss`，所以模型最后一层 `l5` 不需要加任何激活，直接输出 logits。

---

## 4. 训练与测试

**训练循环**（标准五步）：清零梯度 → 前向 → 算损失 → 反向 → 更新。

```python
optimizer.zero_grad()              # 清空上一轮梯度
outputs = model(inputs)            # 前向：得到 (64, 10) 的 logits
loss = criterion(outputs, target)  # 算损失
loss.backward()                    # 反向：算梯度
optimizer.step()                   # 更新权重
```

注意形状：`outputs` 是 `(64, 10)`（每个样本 10 个类别分数），`target` 是 `(64,)`（每个样本一个正确类别编号，如 `7`）。`CrossEntropyLoss` 接受这种类别编号标签，**不需要 one-hot**。

**测试**：

```python
with torch.no_grad():                          # 测试不需要梯度
    _, predicted = torch.max(outputs.data, dim=1)  # 沿类别维取最大值的索引
    correct += (predicted == labels).sum().item()  # 统计预测对的个数
```

- `torch.max(..., dim=1)` 返回 (最大值, 索引)，我们只要**索引**（即预测的类别），所以用 `_` 丢掉最大值。
- 准确率 = 正确数 / 总数。

主循环跑 10 个 epoch，每训练完一轮就在测试集评估一次。这个网络一般能达到 **97-98%** 的准确率。

In [ ]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
 
# prepare dataset
 
batch_size = 64
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]) # 归一化,均值和方差
 
train_dataset = datasets.MNIST(root='../dataset/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root='../dataset/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
 
# design model using class
 
 
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.l1 = torch.nn.Linear(784, 512)
        self.l2 = torch.nn.Linear(512, 256)
        self.l3 = torch.nn.Linear(256, 128)
        self.l4 = torch.nn.Linear(128, 64)
        self.l5 = torch.nn.Linear(64, 10)
 
    def forward(self, x):
        x = x.view(-1, 784)  # -1其实就是自动获取mini_batch
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x))
        x = F.relu(self.l4(x))
        return self.l5(x)  # 最后一层不做激活，不进行非线性变换
 
 
model = Net()
 
# construct loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)
 
# training cycle forward, backward, update
 
 
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        # 获得一个批次的数据和标签
        inputs, target = data
        optimizer.zero_grad()
        # 获得模型预测结果(64, 10)
        outputs = model(inputs)
        # 交叉熵代价函数outputs(64,10),target（64）
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch+1, batch_idx+1, running_loss/300))
            running_loss = 0.0
 
 
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1) # dim = 1 列是第0个维度，行是第1个维度
            total += labels.size(0)
            correct += (predicted == labels).sum().item() # 张量之间的比较运算
    print('accuracy on test set: %d %% ' % (100*correct/total))
 
 
if __name__ == '__main__':
    for epoch in range(10):
        train(epoch)
        test()

 23%|██▎       | 2.33M/9.91M [02:18<08:28, 14.9kB/s]